In [ ]:
import ollama

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures, TrigramCollocationFinder, TrigramAssocMeasures
from sklearn.feature_extraction.text import TfidfVectorizer
from boardgames_recsys.data.filtering import filter_df
import boardgames_recsys.text.filtering as ft
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
from boardgames_recsys.evaluation.ratings import *
from boardgames_recsys.text.llm import *

from surprise import NMF
from surprise import Dataset
from surprise.reader import Reader
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

from IPython.core.display import display_html

import textwrap # to avoid scrolling on long strings in jupyter
#%load_ext autoreload
#%autoreload 2
%matplotlib inline


In [ ]:
folder = "../database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 
rev_filter = filter_df(avis_clean, min_reviews)
games_means = rev_filter[["Game id", "Rating"]].groupby("Game id").mean().reset_index()

rev_filter = rev_filter.assign(index=rev_filter.index)
rev_filter["Length"] = rev_filter["Comment body"].str.split().apply(len)

In [ ]:
lemmas = pd.read_csv("../generated_data/Lemmas_VER_cleaned.csv", index_col=0)
lemmas = lemmas[~lemmas["Lemma"].isna()]
comment_lemmatized = lemmas.groupby("Comment line")["Lemma"].apply(" ".join).reset_index().dropna(subset=["Lemma"])
comment_lemmatized = comment_lemmatized.merge(rev_filter[["Game id", "User id", "index"]], left_on="Comment line", right_on="index").rename(columns={"Lemma":"Comment body"})

In [ ]:
# Embeddings on all games descriptions
# games_filter = jeux_clean[jeux_clean["Game id"].isin(rev_filter["Game id"])].sort_values("Game id")[["Game id", "Description"]]

# from FlagEmbedding import BGEM3FlagModel

# model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

# descriptions = games_filter["Description"].tolist()

# # Batch encode
# encoded = model.encode(
#     descriptions,
#     batch_size=12,         # now this matters
#     max_length=6144,       # truncate long ones
#     return_dense=True,     # default is True, but explicit is good
# )

### Summarizing game/user comments 

#### Number of reviews per user

In [ ]:
users_count = rev_filter.groupby("User id").agg(count=('Game id', 'count'), mean=('Rating', 'mean')).reset_index().sort_values(by="mean").head(30)
games_count = rev_filter.groupby("Game id").agg(count=('User id', 'count'), mean=('Rating', 'mean')).reset_index().sort_values(by="mean").head(30)

users_styler = users_count.style.set_table_attributes("style='display:inline'").set_caption('Users count reviews, mean rating')
games_styler = games_count.style.set_table_attributes("style='display:inline'").set_caption('Users count reviews, mean rating')
    
display_html(users_styler._repr_html_()+games_styler._repr_html_(), raw=True)

### Game's comments

**Game with 140 reviews and mean rating : 6.74**

In [ ]:
game = 4342
game_comments = rev_filter[rev_filter["Game id"] == game]
game_comments = game_comments.assign(Batch=assign_batch_number(game_comments, 2300))
game_comments_batched = game_comments.groupby("Batch")["Comment body"].apply("\n".join).tolist()
print("Nb words:", calc_nb_words(game_comments["Comment body"]))
response = call_model_by_batch(game_comments_batched, "game")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

**Game with 93 reviews and mean ratings : 3.27**

In [ ]:
game = 4360
game_comments = rev_filter[rev_filter["Game id"] == game]
game_comments = game_comments.assign(Batch=assign_batch_number(game_comments, 2300))
game_comments_batched = game_comments.groupby("Batch")["Comment body"].apply("\n".join).tolist()
print("Nb words:", calc_nb_words(game_comments["Comment body"]))
response = call_model_by_batch(game_comments_batched, "game")
print("\n".join(textwrap.fill(word, width=100) for word in response['message']['content'].split('\n')))

### Comments embeddings per cluster

In [ ]:
model = NMF(n_factors=20, random_state=42, biased=False, reg_pu= 0.1, reg_qi= 0.1)
data = Dataset.load_from_df(rev_filter[["User id", "Game id", "Rating"]], reader=Reader(rating_scale=(0, 10)))
trainset = data.build_full_trainset()
nmf = model.fit(trainset)

# Extract matrices
U = nmf.pu  # User-feature matrix (W)
G = nmf.qi  # Item-feature matrix (H)

games_ids = np.array([trainset.to_raw_iid(i) for i in range(len(G))])
users_ids = np.array([trainset.to_raw_uid(u) for u in range(len(U))])
G = G[np.argsort(games_ids), :]

NB_CLUSTERS = 30
kmeans = KMeans(n_clusters=NB_CLUSTERS, random_state=42) 
kmeans.fit(G) 

games_clusters = pd.DataFrame(data={"Game id":np.sort(games_ids), "Cluster":kmeans.labels_})

#### Embeddings on comments per clusters
Only comments that have $10$ to $300$ words are selected

In [ ]:
comment_real = rev_filter[(rev_filter["Length"] >= 10) & (rev_filter["Length"] <= 300)][["User id", "Game id", "index", "Comment body"]]

# comment_lemmatized = lemmas.groupby("Comment line")["Lemma"].apply(" ".join).reset_index().dropna(subset=["Lemma"])
# comment_lemmatized = comment_lemmatized.merge(rev_filter[["Game id", "User id"]], left_on="Comment line", right_index=True)

comment_lemmatized = comment_lemmatized.merge(comment_real[["index"]], left_on="Comment line", right_on="index").sort_values("Comment line")
comment_real = comment_real.merge(comment_lemmatized[["Comment line"]], right_on="Comment line", left_on="index").sort_values("index")

comment_lemmatized.shape, comment_real.shape

In [ ]:
desc_lemmas = pd.read_csv("../generated_data/desc_lemmas.csv", index_col=0)

# Errors in lemmatization
desc_lemmas.loc[(desc_lemmas["Lemma"] == "poindre") & (desc_lemmas["Tokens"] == "points"), ["Lemma", "POS"]] = ["point", "NOM"] 
desc_lemmas.loc[(desc_lemmas["Lemma"] == "poindre") & (desc_lemmas["Tokens"] == "poins"), ["Lemma", "POS"]] = ["poin", "NOM"] 
desc_lemmas.loc[(desc_lemmas["Lemma"] == "poindre") & (desc_lemmas["Tokens"] == "pointes"), ["Lemma", "POS"]] = ["pointes", "NOM"] 

desc_lemmatized = desc_lemmas.groupby("Game id")["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma":"Description"})

desc_embeddings = pd.read_parquet("../generated_data/descriptions_embeddings.parquet")

# Delete games that have no description
desc_embeddings = desc_embeddings[~desc_embeddings["Description"].str.contains("Aucune description")]
desc_embeddings["Length"] = desc_embeddings["Description"].str.split().apply(len)

# Find intersection
desc_lemmatized = desc_lemmatized[desc_lemmatized["Game id"].isin(desc_embeddings["Game id"])]
desc_lemmatized.shape, desc_embeddings.shape

### TF-IDF is tried -> works bad

In [ ]:
# bigrams = BigramCollocationFinder.from_documents(desc_lemmatized["Description"].str.split().tolist())
# bigrams_freq = bigrams.score_ngrams(BigramAssocMeasures.raw_freq)

# bigrams_df = pd.DataFrame(data=[list(info) for info in bigrams_freq])

# bigrams_df[0] = bigrams_df[0].apply(list).apply(" ".join)
# bigrams_df = bigrams_df.rename(columns={0:"Lemma", 1:"Freq"})

# bigrams_df = bigrams_df[~bigrams_df["Lemma"].str.contains("joueur")]

# sns.set_theme(rc={"figure.figsize":(7, 15)})
# sns.barplot(data=bigrams_df.sort_values("Freq").tail(80), x="Freq", y="Lemma")

In [ ]:
# count = desc_lemmas["Lemma"].value_counts().reset_index().sort_values(by="count", ascending=False)
# corpus = count[count["count"] >= 3]["Lemma"].values
# desc_lemmas = desc_lemmas[desc_lemmas["Lemma"].isin(corpus)]

# # desc_lemmas = desc_lemmas[desc_lemmas["Game id"].isin(count.loc[count["Lemma"] >= 10, "Game id"])]

# desc_lemmatized = desc_lemmas.groupby("Game id")["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma":"Description"})
# desc_lemmatized.shape

In [ ]:
# vectorizer = TfidfVectorizer(ngram_range=(2, 2)) # tfidf on bigrams
# desc_tfidf = vectorizer.fit_transform(desc_lemmatized["Description"])

# # Convert back to description
# games, bigrams = desc_tfidf.nonzero()
# tfidf = desc_tfidf.data
# ptr = desc_tfidf.indptr
# corpus = vectorizer.get_feature_names_out()

# threshold = 0.1

# bigrams_df = pd.DataFrame(data={"index":games, 
#                                 "Bigrams":[corpus[b] for b in bigrams], 
#                                 "tdidf":tfidf})

# games_map = pd.DataFrame(data={"index": np.arange(0, desc_lemmatized.shape[0]), 
#                                "Game id" : desc_lemmatized["Game id"]})

# bigrams_df = bigrams_df[bigrams_df["tdidf"] > threshold]
# games_bigrams = bigrams_df.merge(games_map, on="index")
# games_filter = games_bigrams.groupby("Game id")["Bigrams"].apply(" ".join).reset_index().rename(columns={"Bigrams":"Description"})
# games_filter
# games_filter_real = jeux_clean[jeux_clean["Game id"].isin(games_filter["Game id"])][["Game id", "Description"]]

In [ ]:
def find_represent_comment(cluster:int, games_clusters:pd.DataFrame, text_lemmatized:pd.DataFrame, 
                           text_real_embeddings:pd.DataFrame, column_name:str):
    # all comments per cluster
    games_in_cluster = games_clusters[games_clusters["Cluster"] == cluster]
    text_real_embeddings = text_real_embeddings[text_real_embeddings["Game id"].isin(games_in_cluster["Game id"])].sort_values("Game id")
    text_lemmatized = text_lemmatized[text_lemmatized["Game id"].isin(games_in_cluster["Game id"])].sort_values("Game id")

    #print(real_text, text_lemmatized)
    print("Nb texts :", text_real_embeddings.shape[0], text_lemmatized.shape[0])

    # Extract embeddings
    embeddings = np.array(text_real_embeddings["Embedding"].tolist())
    
    # Find mean comment
    mean_text = embeddings.mean(axis=0) 

    # Find closest existing comment 
    dist = euclidean_distances(mean_text.reshape(1, -1), embeddings)
    #print(dist.shape)
    return text_real_embeddings, text_lemmatized, embeddings, dist.flatten()

def plot_bigrams(text_selected, ax=None):
    bigrams = BigramCollocationFinder.from_documents(text_selected.str.split().tolist())
    bigrams_freq = bigrams.score_ngrams(BigramAssocMeasures.raw_freq)

    bigrams_df = pd.DataFrame(data=[list(info) for info in bigrams_freq])

    bigrams_df[0] = bigrams_df[0].apply(list).apply(" ".join)
    bigrams_df = bigrams_df.rename(columns={0:"Lemma", 1:"Freq"})

    if ax is not None:
        ax = sns.barplot(data=bigrams_df.sort_values(by="Freq", ascending=False).head(40), x="Freq", y="Lemma", ax=ax)
    else:
        sns.set_theme(rc={"figure.figsize":(15, 6)})
        ax = sns.barplot(data=bigrams_df.sort_values(by="Freq", ascending=False).head(90), y="Freq", x="Lemma")
        ax.set_xticks(ax.get_xticks())  # Explicitly set tick locations
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90); # ; to avoid printing return value
        ax.set_title(f"Nb texts selected : {text_selected.size}")
    
    return bigrams_df
def plot_bigrams_both(comments_selected, desc_selected):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8))
    plot_bigrams(comments_selected, ax1)
    ax1.set_title("Comments bigrams")

    plot_bigrams(desc_selected, ax2)
    ax2.set_title("Descriptions bigrams")

    plt.tight_layout()

def describe_cluster(cluster, perc_desc, perc_comments, llm_k, games_clusters, desc_lemmatized, desc_real,
                     comments_lemmatized, comments_real, comments=False):

    real_desc_cluster, lemmas_desc_cluster, embeddings, dist_desc = find_represent_comment(cluster, games_clusters,
                                                                                      desc_lemmatized, desc_real, "Description")
    print("Description embeddings completed")

    # Treat descriptions
    dist_min_desc = np.min(dist_desc)
    closest_desc = np.argwhere(dist_desc <= dist_min_desc * perc_desc).flatten()
    #closest_desc = np.argpartition(dist_desc.flatten(), 10)[:10]
    
    # Treat comments
    if comments:
        real_comments_cluster, lemmas_comments_cluster, embeddings, dist_comments = find_represent_comment(cluster, games_clusters,
                                                                             comments_lemmatized, comments_real, "Comment body")
        print("Comments embeddings completed")

        dist_min_comments = np.min(dist_comments)
        closest_comments = np.argwhere(dist_comments <= dist_min_comments * perc_comments).flatten()

        print('\033[1m' + "CENTER COMMENT :" + '\033[0m', textwrap.fill(real_comments_cluster.iloc[np.argmin(dist_comments)]["Comment body"], width=100))

        print(real_desc_cluster.shape, real_comments_cluster.shape, dist_desc.shape, dist_comments.shape)
    desc_for_llm = real_desc_cluster.iloc[np.argpartition(dist_desc.flatten(), llm_k)[:llm_k]]
    if comments:
        plot_bigrams_both(lemmas_comments_cluster.iloc[closest_comments]["Comment body"], lemmas_desc_cluster.iloc[closest_desc]["Description"])
    else:
        print("Distance descriptions : ", np.sort(dist_desc)[:llm_k])
        ret2 = plot_bigrams(lemmas_desc_cluster.iloc[closest_desc]["Description"])

    print('\033[1m' + "CENTER DESC :" + '\033[0m', textwrap.fill(real_desc_cluster.iloc[np.argmin(dist_desc)]["Description"], width=100), "\n")

    return desc_for_llm, ret2

def plot_bigrams_tfidf(games_bigrams, games):
    games_bigrams = games_bigrams[games_bigrams["Game id"].isin(games)]

    bigrams_df = games_bigrams.groupby("Bigrams")["Game id"].nunique().reset_index(name="Freq")
    # bigrams_games_count = games_bigrams.groupby("Bigrams")["Game id"].nunique().reset_index(name="Games Appeared")

    # bigrams_df = pd.merge(bigrams_freq, bigrams_games_count, on="Bigrams")

    # bigrams_df["Freq"] = bigrams_df["Freq"]


    sns.set_theme(rc={"figure.figsize":(15, 6)})
    ax = sns.barplot(data=bigrams_df.sort_values(by="Freq", ascending=False).head(100), y="Freq", x="Bigrams")
    ax.set_xticks(ax.get_xticks())  # Explicitly set tick locations
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90); # ; to avoid printing return value

#### Cluster 5

**Thème commun : La conquête et le pouvoir**

Unigrams/Bigrams:
- acquerir suprémacie
- phase attaquer
- pétrole
- rail
- proteger monde
- bombe atomique
- six superpuissance

**Targeted public : adults**

In [ ]:
desc_embeddings["Embedding"].iloc[0].shape

In [ ]:
desc_llm, bigrams = describe_cluster(5, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
bigrams.sort_values("Freq", ascending=False).iloc[100:150]
lemmas = ["petroler minerai", "aventurier rail", "carte armee", "conquerir planete", "phase commerce", "personnage secret",
          "bataille epique", "jeu codificateur", "six superpuissance", "prendre possession"]

df = bigrams[bigrams["Lemma"].isin(lemmas)]
# df
# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print("Nb tokens sent :", response["prompt_eval_count"])
# print(response["message"]["content"])

### Cluster 0

**Focus sur l'expansion, la construction et la gestion de ressources**

Unigrams/bigrams
- tuile | alimenter maison 
- alimenter eau
- statuette
- coût construction

**Public**
- Jeunes, famille

In [ ]:
desc_llm, bigrams = describe_cluster(0, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
 
lemmas = ["alimenter eau", "lieu culte", "tuile maison", "maison alimenter", "navire adverse",
            "statuette dieu", "case empire", "cout construction", "lieu vente", "ville region"]
df = pd.concat([df, bigrams[bigrams["Lemma"].isin(lemmas)]])
# df = pd.concat([df, desc_llm[desc_llm["Lemma"].isin(bigrams)]])
# df
#print("Games selected", cluster_desc.shape[0])
# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])

### Cluster 1

**Themes**
- Collection des ressources (trésors), exploration

In [ ]:
desc_llm, bigrams = describe_cluster(1, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
lemmas=["prendre batiment", "reservoir stockage", "carte sphinx", "carte porcelaine", "prix vente", "salle tresor", 
           "tueur gage", "carte marchandise", "falloir attraper", "relic runners"]
df = pd.concat([df, bigrams[bigrams["Lemma"].isin(lemmas)]])

# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# print("Games selected", cluster_desc.shape[0])
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])

### Cluster 2

**Complexité**
- jeux qui peuvent être **longs** (30min à plusieurs heures) avec des règles **complexes**

In [ ]:
desc_llm, bigrams = describe_cluster(2, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
bigrams.sort_values("Freq", ascending=False).iloc[0:50]
# #bigrams=["face cacher", "carte richesse", "carter politique", "livret regle", "paquet carte", "premier acheteur",
# #       "ordre tour", "objectif secret", "politique militaire", "faire commerce"]

# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])


In [ ]:
#df["Cluster"] = [5] * 10 + [0] * 10 + [1] * 10 + [25] * 10
df = df.sort_values("Freq", ascending=False)
df
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for i, cluster in enumerate(np.sort(df["Cluster"].unique())):
    bigrams = df[df["Cluster"] == cluster]
    sns.barplot(data=bigrams, x="Freq", y="Lemma", ax=axes[i])
    axes[i].set_ylabel("")
    axes[i].set_xlabel("Frequency")
    axes[i].set_title(f"Cluster {cluster}")

plt.tight_layout()
fig.savefig("../images/bigrams.svg", format="svg", bbox_inches="tight")

### Cluster 3 

Capture territoire

In [ ]:
desc_llm, bigrams = describe_cluster(3, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
bigrams.sort_values("Freq", ascending=False).iloc[50:100]
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 4 --> logique, déduction


Superbes visuels

In [ ]:
desc_llm, bigrams = describe_cluster(4, 1.25, 1.25, 15, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 6 

Culture | fantaisie

In [ ]:
desc_llm, bigrams = describe_cluster(6, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 7 

Culture | Fantaisie


In [ ]:
desc_llm, bigrams = describe_cluster(7, 1.25, 1.25, 15, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])

cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 8

Superbes visuels

In [ ]:
desc_llm, bigrams = describe_cluster(8, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 9

Longs & complexes

In [ ]:
desc_llm, bigrams = describe_cluster(9, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 10 -> worst rated

In [ ]:
# desc_llm = describe_cluster(10, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_real, comment_lemmatized, comment_real, comments=False)
# #print("Games selected", cluster_desc.shape[0])
# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])

### Cluster 11 

Capture territoire

In [ ]:
desc_llm, bigrams = describe_cluster(11, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 12

De type Civilisation

In [ ]:
desc_llm, bigrams = describe_cluster(12, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)

cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 13

Longs & complexes

In [ ]:
desc_llm, bigrams = describe_cluster(13, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 14

Rapide & tactique

In [ ]:
desc_llm, bigrams = describe_cluster(14, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 15

Eurogames

In [ ]:
desc_llm, bigrams = describe_cluster(15, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 16

Eurogames

In [ ]:
desc_llm, bigrams = describe_cluster(16, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 17

Collecte

In [ ]:
desc_llm, bigrams = describe_cluster(17, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 18

De type Civilisation

In [ ]:
desc_llm, bigrams = describe_cluster(18, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])

cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])
bigrams.sort_values(by="Freq", ascending=False).iloc[50:100]


### Cluster 19 

Construction & expension

In [ ]:
desc_llm, bigrams = describe_cluster(19, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 20

Rapide & tactique

In [ ]:
desc_llm, bigrams = describe_cluster(20, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 21 

De type Civilisation

In [ ]:
desc_llm, bigrams = describe_cluster(21, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 22

Logique & déduction


In [ ]:
desc_llm, bigrams = describe_cluster(22, 1.25, 1.25, 7, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 23 

Logique & déduction

In [ ]:
desc_llm, bigrams = describe_cluster(23, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 24 (best rated)

In [ ]:
desc_llm, bigrams = describe_cluster(24, 1.10, 1.10, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "desc_summary")
print(response["message"]["content"])

### Cluster 25 

Longs & complexes

In [ ]:
desc_llm, bigrams = describe_cluster(25, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
lemmas=["twilight imperium", "sherlock holmes", "blake mortimer", "carte invention", "grand ancien",
        "grand nombre", "livret regle", "jeu cooperatif", "univers warhammer", "plan carriere"]
df = pd.concat([df, bigrams[bigrams["Lemma"].isin(lemmas)]])

#bigrams.sort_values("Freq", ascending=False).iloc[0:50]
# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])

### Cluster 26 -> close to best rated cluster

Collecte

In [ ]:
desc_llm, bigrams = describe_cluster(26, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
# cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
# cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

# print("Nb words:", calc_nb_words(cluster_desc["Description"]))
# response=call_model_by_batch(cluster_desc_batched, "descriptions")
# print(response["message"]["content"])
bigrams.sort_values("Freq", ascending=False).iloc[100:150]

### Cluster 27 

Capture territoire

In [ ]:
desc_llm, bigrams = describe_cluster(27, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 28 

Culture | Fantaisie

In [ ]:
desc_llm, bigrams = describe_cluster(28, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)
#print("Games selected", cluster_desc.shape[0])
cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])

### Cluster 29 

Longs & complexes


In [ ]:
desc_llm, bigrams = describe_cluster(29, 1.25, 1.25, 10, games_clusters, desc_lemmatized, desc_embeddings, comment_lemmatized, comment_real, comments=False)

cluster_desc = desc_llm.assign(Batch=assign_batch_number(desc_llm, 2000))
cluster_desc_batched = cluster_desc.groupby("Batch")["Description"].apply("\n".join).tolist()

print("Nb words:", calc_nb_words(cluster_desc["Description"]))
response=call_model_by_batch(cluster_desc_batched, "descriptions")
print(response["message"]["content"])